# Dataset
Many different collections of code/commit related datasets available here: https://huggingface.co/collections/smcleish/diff-datasets

Chose to use CommitPackFT, and only the Python split for this project for consistency: https://huggingface.co/datasets/bigcode/commitpackft

In [ ]:
# Pre-requisites
!pip install -U "datasets<3.0.0"
!pip install -q pandas tqdm
!pip install groq
!pip install -q sentence-transformers faiss-cpu

In [ ]:
import random
import os
import json
import pandas as pd
import numpy as np
import difflib
import faiss
from datasets import load_dataset
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from groq import Groq
from google.colab import userdata
from google.colab import files

In [ ]:
# Load only the Python subset
ds = load_dataset("bigcode/commitpackft", "python", split="train", trust_remote_code=True)

print(ds)
print(ds.column_names)

In [ ]:
# Convert to DataFrame + Inspect
df = ds.to_pandas()
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Generate Unified Diff Text
def make_unified_diff(row, max_lines=None):
    """
      Generates a standard unified diff string from old and new file contents.

      Args:
          row (pd.Series): A dataframe row containing 'old_file', 'new_file',
                            'old_contents', and 'new_contents'.
          max_lines (int, optional): The maximum number of lines to return to
                                      prevent context window overflow. Defaults to None.

      Returns:
          str: The formatted unified diff.
    """
    old_file = row.get("old_file", "old_file.py")
    new_file = row.get("new_file", "new_file.py")
    old_text = row.get("old_contents", "") or ""
    new_text = row.get("new_contents", "") or ""

    diff_lines = list(difflib.unified_diff(
        old_text.splitlines(),
        new_text.splitlines(),
        fromfile=old_file,
        tofile=new_file,
        lineterm=""
    ))

    if max_lines is not None:
        diff_lines = diff_lines[:max_lines]

    return "\n".join(diff_lines)

In [ ]:
df["diff"] = df.apply(make_unified_diff, axis=1)

display(df[["old_file", "new_file", "subject", "message", "diff"]].head())
print(df["diff"].iloc[0][:1500])

In [ ]:
# Basic Data Exploration
df["diff_chars"] = df["diff"].str.len()
df["subject_chars"] = df["subject"].fillna("").str.len()
df["message_chars"] = df["message"].fillna("").str.len()

display(df[["diff_chars", "subject_chars", "message_chars"]].describe())

print("Empty diffs:", (df["diff_chars"] == 0).sum())
print("Empty subjects:", (df["subject"].fillna("").str.strip() == "").sum())
print("Duplicate diffs:", df["diff"].duplicated().sum())
print("Duplicate subjects:", df["subject"].duplicated().sum())

## Observations from Data Exploration
1. Mean diff length of 781 is good, but max of 4257 is very large, may need truncation or filtering, else retrieval may degrade
2. 52 Empty diffs, to remove
3. Subject is 44 chars avg, which Message is longer at 68, but can go up to 3k+. Decided to use subject as target output as they are closer to real commit title (Or use both? To decide later)

Decided to filter extreme diff outliers (>3000 chars) and truncate remaining diffs to ~1000 chars.

In [ ]:
# 1. Remove empty diffs
df = df[df["diff"].str.len() > 0]

# 2. Filter extremely long diffs (outliers)
df = df[df["diff_chars"] < 3000]

# 3. Truncate diffs (for consistency)
MAX_DIFF_CHARS = 1000
df["diff"] = df["diff"].str[:MAX_DIFF_CHARS]

# 4. Keep only necessary columns
df = df[["diff", "subject"]].copy()
df.rename(columns={"subject": "commit_message"}, inplace=True)

# 5. Reset index
df.reset_index(drop=True, inplace=True)

print("Final dataset shape:", df.shape)
display(df.head())

In [ ]:
# Split processed dataset into retrieval database and held-out evaluation set
train_df, eval_df = train_test_split(
    df,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

print("Retrieval dataset size:", train_df.shape)
print("Evaluation dataset size:", eval_df.shape)

display(train_df.head())
display(eval_df.head())

# Scoring/Retrieval
Two options:
1. Fast baseline only (TF-IDF)
2. TF-IDF + embeddings (better but slightly heavier)

### TF-IDF Baseline

In [ ]:
# Use the cleaned train_df with columns: diff, commit_message
corpus = train_df["diff"].fillna("").tolist()

vectorizer = TfidfVectorizer(
    max_features=50000,
    lowercase=False,
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2),
    min_df=2
)

tfidf_matrix = vectorizer.fit_transform(corpus)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

In [ ]:
# Retrieval Function With Exclusion
def retrieve_similar_diffs(query_diff, top_k=5, exclude_idx=None):
    """
        Retrieves the most similar historical commits using TF-IDF and Cosine Similarity.

        Args:
            query_diff (str): The target git diff to find matches for.
            top_k (int): The number of similar examples to retrieve.
            exclude_idx (int, optional): The index to ignore (used during hold-out evaluation
                                        to prevent the model from retrieving the exact answer).

        Returns:
            pd.DataFrame: A dataframe containing the similarity scores, original diffs,
                          and human-written commit messages of the top-k matches.
    """
    query_vec = vectorizer.transform([query_diff])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    if exclude_idx is not None:
        scores[exclude_idx] = -1

    top_indices = scores.argsort()[::-1][:top_k]

    results = train_df.iloc[top_indices].copy()
    results["similarity_score"] = scores[top_indices]

    return results[["similarity_score", "commit_message", "diff"]]

### TF-IDF + embeddings

In [ ]:
# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# Prepare diff texts for embedding
diff_texts = train_df["diff"].fillna("").tolist()

print("Number of diffs:", len(diff_texts))
print("Sample diff:")
print(diff_texts[0][:500])

In [ ]:
# Encode all diffs
# As we normalized embeddings, inner product search becomes equivalent to cosine similarity
embeddings = model.encode(
    diff_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

In [ ]:
# Build FAISS index
embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print("Number of vectors in index:", index.ntotal)

In [ ]:
# Retrieval Function
def retrieve_similar_diffs_embedding(query_diff, top_k=5, exclude_idx=None):
    """
      Retrieves the most similar historical commits using semantic dense embeddings (FAISS).

      Args:
          query_diff (str): The target git diff to find matches for.
          top_k (int): The number of similar examples to retrieve.
          exclude_idx (int, optional): The index to ignore during hold-out evaluation.

      Returns:
          pd.DataFrame: A dataframe containing the similarity scores, original diffs,
                        and human-written commit messages of the top-k matches.
    """
    query_embedding = model.encode(
        [query_diff],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    search_k = top_k + 1 if exclude_idx is not None else top_k
    scores, indices = index.search(query_embedding, search_k)

    scores = scores[0]
    indices = indices[0]

    results = []

    for score, idx in zip(scores, indices):
        if exclude_idx is not None and idx == exclude_idx:
            continue

        results.append({
            "similarity_score": float(score),
            "commit_message": train_df.iloc[idx]["commit_message"],
            "diff": train_df.iloc[idx]["diff"]
        })

        if len(results) == top_k:
            break

    return pd.DataFrame(results)

In [ ]:
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)


def build_commit_prompt(current_diff, retrieved_examples_df):
    examples_text = ""

    for example_number, row in enumerate(retrieved_examples_df.itertuples(), start=1):
        examples_text += f"""
          Example {example_number}
          Similarity score: {getattr(row, "similarity_score", "N/A")}

          Diff:
          {row.diff}

          Human commit message:
          {row.commit_message}
          ---
        """

    prompt = f"""
      You are an assistant that writes high-quality Git commit messages.

      Task:
      Given a current code diff, generate a concise and accurate commit message.

      Guidelines:
      - Use imperative mood, e.g. "Add", "Fix", "Update", "Refactor"
      - Keep the main message under 72 characters if possible
      - Focus on the purpose of the change, not every line changed
      - Do not mention files unless necessary
      - Do not invent behavior that is not supported by the diff
      - Use the retrieved examples only as style/context references

      Retrieved similar examples:
      {examples_text}

      Current diff:
      {current_diff}

      Return your answer as valid JSON with this exact structure:
      {{
        "commit_message": "...",
        "commit_type": "feature | fix | refactor | docs | test | style | chore | other",
        "confidence": "high | medium | low",
        "reasoning_summary": "Briefly explain why this commit message fits the diff."
      }}
    """

    return prompt.strip()


def generate_smart_commit(current_diff, retriever_function, top_k=5, model_name="llama-3.1-8b-instant"):
    """
    Orchestrates the Retrieval-Augmented Generation (RAG) pipeline to create a commit message.

    Args:
        current_diff (str): The current uncommitted code changes.
        retriever_function (callable): The function used to fetch context
                                       (e.g., retrieve_similar_diffs_embedding).
        top_k (int): The number of examples to include in the few-shot prompt.
        model_name (str): The Groq LLM model to query.

    Returns:
        dict: A structured dictionary containing the generated commit message,
              inferred commit type, model confidence, reasoning, and the retrieved context.
    """
    retrieved_examples_df = retriever_function(current_diff, top_k=top_k)

    prompt = build_commit_prompt(
        current_diff=current_diff,
        retrieved_examples_df=retrieved_examples_df
    )

    completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You generate concise, accurate Git commit messages from code diffs."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        model=model_name,
        temperature=0.2,
        response_format={"type": "json_object"}
    )

    raw_output = completion.choices[0].message.content

    try:
        parsed_output = json.loads(raw_output)
    except json.JSONDecodeError:
        parsed_output = {
            "commit_message": raw_output.strip(),
            "commit_type": "other",
            "confidence": "low",
            "reasoning_summary": "Model output was not valid JSON."
        }

    analysis_result = {
        "input_diff": current_diff,
        "generated_commit_message": parsed_output.get("commit_message", ""),
        "commit_type": parsed_output.get("commit_type", "other"),
        "confidence": parsed_output.get("confidence", "low"),
        "reasoning_summary": parsed_output.get("reasoning_summary", ""),
        "retrieved_examples": retrieved_examples_df,
        "prompt": prompt,
        "raw_model_output": raw_output
    }

    return analysis_result

In [ ]:
test_idx = np.random.choice(eval_df.index)

current_diff = eval_df.loc[test_idx, "diff"]
reference_message = eval_df.loc[test_idx, "commit_message"]

result = generate_smart_commit(
    current_diff=current_diff,
    retriever_function=retrieve_similar_diffs_embedding,
    top_k=5
)

print("Input diff:")
print(result["input_diff"])

print("\nReference message:")
print(reference_message)

print("\nGenerated message:")
print(result["generated_commit_message"])

print("\nCommit type:")
print(result["commit_type"])

print("\nConfidence:")
print(result["confidence"])

print("\nReasoning:")
print(result["reasoning_summary"])

print("\nRetrieved examples:")
display(result["retrieved_examples"])

# Evaluation

In [ ]:
def compute_simple_retrieval_score_summary(eval_df, sample_size=30, top_k=3, random_seed=42):
    random.seed(random_seed)

    sample_size = min(sample_size, len(eval_df))
    sampled_indices = random.sample(range(len(eval_df)), sample_size)

    rows = []

    for eval_idx in sampled_indices:
        query_diff = eval_df.loc[eval_idx, "diff"]
        reference_message = eval_df.loc[eval_idx, "commit_message"]

        tfidf_results = retrieve_similar_diffs(
            query_diff,
            top_k=top_k
        )

        embedding_results = retrieve_similar_diffs_embedding(
            query_diff,
            top_k=top_k
        )

        for method_name, results_df in [
            ("TF-IDF", tfidf_results),
            ("Embedding + FAISS", embedding_results)
        ]:
            scores = results_df["similarity_score"].astype(float).tolist()

            rows.append({
                "eval_index": eval_idx,
                "method": method_name,
                "reference_message": reference_message,
                "top1_similarity": scores[0],
                "top3_average_similarity": np.mean(scores),
                "top1_retrieved_message": results_df.iloc[0]["commit_message"],
                "top2_retrieved_message": results_df.iloc[1]["commit_message"] if len(results_df) > 1 else "",
                "top3_retrieved_message": results_df.iloc[2]["commit_message"] if len(results_df) > 2 else "",
            })

    score_df = pd.DataFrame(rows)

    summary_df = (
        score_df
        .groupby("method")[["top1_similarity", "top3_average_similarity"]]
        .mean()
        .reset_index()
    )

    return score_df, summary_df

In [ ]:
retrieval_score_df, retrieval_summary_df = compute_simple_retrieval_score_summary(
    eval_df=eval_df,
    sample_size=30,
    top_k=3,
    random_seed=42
)

display(retrieval_summary_df)
display(retrieval_score_df.head())

In [ ]:
qualitative_sample_df = retrieval_score_df.copy()

qualitative_sample_df = qualitative_sample_df[
    [
        "eval_index",
        "method",
        "reference_message",
        "top1_similarity",
        "top3_average_similarity",
        "top1_retrieved_message",
        "top2_retrieved_message",
        "top3_retrieved_message"
    ]
]

qualitative_sample_df.to_csv(
    "retrieval_qualitative_sample.csv",
    index=False
)